In [ ]:
### NOTE ###
# The analyses reported in the paper were conducted using Japanese prompts.
# Due to variability in the LLM’s responses, rerunning the analysis will not reproduce exactly the same results.
# The data used in the paper are provided in "results/fig5_photos_va_results_paper.csv".

In [ ]:
from openai import OpenAI
import json
import pandas as pd
from pathlib import Path
import base64
import io
from PIL import Image

In [ ]:
client = OpenAI()
model = "gpt-5.2"

photo_va_results_jp = "results/fig5_photos_va_results_test_jp.csv"
photo_va_results_en = "results/fig5_photos_va_results_test_en.csv"

img_root = Path("../database/final_database")

image_paths = sorted(
    [p for p in img_root.rglob("*") if p.suffix.lower() == ".jpg"]
)
print(f"images found: {len(image_paths)}")

In [ ]:
# analysis in Japanese (paper)

def create_message(image_path):
    system_message="""
    あなたは感情分析の専門家です。
    与えられた表情の写真のときの、valenceの値とarousalの値を出してください。
    valenceは-1（不快）～1（快）の範囲で出してください。
    arousalは-1（非覚醒）～1（覚醒）の範囲で出してください。
    どちらも小数点以下2桁で答えてください。
    """

    # resize image to reduce token
    with Image.open(image_path) as img:
        img.thumbnail((320, 240))

    buffer = io.BytesIO()
    img.save(buffer, format="JPEG")

    encoded_image = base64.b64encode(buffer.getvalue()).decode("utf-8")

    user_message = {"type": "image_url", 
                    "image_url": {
                        "url": f'data:image/jpeg;base64,{encoded_image}',
                        "detail": "low"}
                   } 
    input_entry = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": [user_message]}
    ]

    return input_entry

tools = [{
    "type": 'function',
    "function": {
        "name": "analyze_valence_arousal",
        "description": "表情のvalenceの値及びarousalの値を出力する",
        "parameters": {
            "type": "object",
            "properties": {
                "valence": {"type": "number", "minimum": -1, "maximum": 1,
                            "description": "与えられた表情のときのvalenceの値。-1が不快で1が快。小数点以下2桁で回答。"},
                "arousal": {"type": "number", "minimum": -1, "maximum": 1,
                            "description": "与えられた表情のときのarousalの値。-1が非覚醒で1が覚醒。小数点以下2桁で回答。"},
                "reasoning": {"type": 'string',
                              "description": "判断した理由を日本語で1,2文で答えてください。"},
            },
            "required": ["valence", "arousal", "reasoning"],
            "additionalProperties": False
        }},
    "strict": True
}]

processed_image_paths = set()
if Path(photo_va_results_jp).exists():
    existing_df = pd.read_csv(photo_va_results_jp, encoding="utf-8-sig")
    processed_image_paths = set(path for path in existing_df["image_path"])

for num, image in enumerate(image_paths):
    print(num, image)

    # skip images that are already recorded in the CSV file
    if str(image) in processed_image_paths:
        print(f"skip: {image}")
        continue
    
    input_entry = create_message(image)
    image_results = []

    for i in range(10):
        getresponse=True
        while getresponse:
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=input_entry,
                    tools=tools,
                    tool_choice="required",
                )
                response_output = json.loads(response.choices[0].message.tool_calls[0].function.arguments)
                
                image_results.append({
                    "image_path": str(image),
                    "label": image.parent.name,
                    "image_name": image.name,
                    "try_num": f"{i+1}",
                    "valence": response_output["valence"],
                    "arousal": response_output["arousal"],
                    "reasoning": response_output["reasoning"]
                })
                getresponse=False

            except Exception as e:
                print(e)

    # save to csv
    image_df = pd.DataFrame(image_results)

    image_df.to_csv(
        photo_va_results_jp,
        mode="a",
        header=not Path(photo_va_results_jp).exists(),
        index=False,
        encoding="utf-8-sig",
    )

    print(f"saved: {image} ({len(image_results)} results)")

print("all saved:", photo_va_results_jp)

In [ ]:
# analysis in English

def create_message(image_path):
    system_message="""
    You are an expert in emotion analysis.
    For the given facial expression image, provide valence and arousal values.
    The valence value must range from -1 (unpleasant) to 1 (pleasant).
    The arousal value must range from -1 (low arousal) to 1 (high arousal).
    Provide both values rounded to two decimal places.
    """

    # resize image to reduce token
    with Image.open(image_path) as img:
        img.thumbnail((320, 240))

    buffer = io.BytesIO()
    img.save(buffer, format="JPEG")

    encoded_image = base64.b64encode(buffer.getvalue()).decode("utf-8")

    user_message = {"type": "image_url", 
                    "image_url": {
                        "url": f'data:image/jpeg;base64,{encoded_image}',
                        "detail": "low"}
                   } 
    input_entry = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": [user_message]}
    ]

    return input_entry

tools = [{
    "type": 'function',
    "function": {
        "name": "analyze_valence_arousal",
        "description": "Output the valence and arousal values for the given facial expression image.",
        "parameters": {
            "type": "object",
            "properties": {
                "valence": {"type": "number", "minimum": -1, "maximum": 1,
                            "description": "The valence value for the given emotion label. -1 represents unpleasantness, and 1 represents pleasantness. Provide the value rounded to two decimal places."},
                "arousal": {"type": "number", "minimum": -1, "maximum": 1,
                            "description": "The arousal value for the given emotion label. -1 represents low arousal, and 1 represents high arousal. Provide the value rounded to two decimal places."},
                "reasoning": {"type": 'string',
                              "description": "Explain the reasoning behind your assessment in one or two sentences in English."},
            },
            "required": ["valence", "arousal", "reasoning"],
            "additionalProperties": False
        }},
    "strict": True
}]

processed_image_paths = set()
if Path(photo_va_results_en).exists():
    existing_df = pd.read_csv(photo_va_results_en, encoding="utf-8-sig")
    processed_image_paths = set(path for path in existing_df["image_path"])

for num, image in enumerate(image_paths):
    print(num, image)

    # skip images that are already recorded in the CSV file
    if str(image) in processed_image_paths:
        print(f"skip: {image}")
        continue
    
    input_entry = create_message(image)
    image_results = []

    for i in range(10):
        getresponse=True
        while getresponse:
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=input_entry,
                    tools=tools,
                    tool_choice="required",
                )
                response_output = json.loads(response.choices[0].message.tool_calls[0].function.arguments)
                
                image_results.append({
                    "image_path": str(image),
                    "label": image.parent.name,
                    "image_name": image.name,
                    "try_num": f"{i+1}",
                    "valence": response_output["valence"],
                    "arousal": response_output["arousal"],
                    "reasoning": response_output["reasoning"]
                })
                getresponse=False

            except Exception as e:
                print(e)

    # save to csv
    image_df = pd.DataFrame(image_results)

    image_df.to_csv(
        photo_va_results_en,
        mode="a",
        header=not Path(photo_va_results_en).exists(),
        index=False,
        encoding="utf-8-sig",
    )

    print(f"saved: {image} ({len(image_results)} results)")

print("all saved:", photo_va_results_en)